In [1]:
import torch
import json
from torch.utils.data import DataLoader

from models.blip import blip_decoder
from finetune_data import create_dataset

In [2]:
config = {
    "image_size": 384,
    "vit": "base",
    "vit_grad_ckpt": False,
    "vit_ckpt_layer": 0,
    "image_root": "../dataset/flickr30k/images",
    "ann_root": "../dataset/flickr30k",
    "filename": None,
    "max_length": 35,
    "min_length": 5,
    "num_beams": 3,
    "batch_size": 16
}
filenames = [
    "dbi_val.json",
    "edu_val.json",
    "fi_val.json",
    "idc_val.json",
    "mi_val.json",
    "resume_val.json"
]
prompts = [
    "an image of",
    "a photo of",
    "a snapshot of"
]
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

In [3]:
def get_adversarial_output(pretrained_model, config, filenames, device):
    for idx, model_path in enumerate(pretrained_model):
        print(model_path)
        scenario_result = []
        for idj, prompt in enumerate(prompts):
            model = blip_decoder(pretrained=model_path, image_size=config['image_size'], vit=config['vit'], vit_grad_ckpt=config['vit_grad_ckpt'], vit_ckpt_layer=config['vit_ckpt_layer'], prompt=prompt)
            model = model.to(device)
            config["filename"] = filenames[idx]
            dataset = create_dataset("caption_adversarial", config)
            dataloader = DataLoader(dataset, batch_size=config['batch_size'], drop_last=False)
            model.eval()
            result = []
            for image, image_id in dataloader:
                image = image.to(device)
                captions = model.generate(image, sample=False, num_beams=config['num_beams'], max_length=config['max_length'], min_length=config['min_length'])
                for caption, img_id in zip(captions, image_id):
                    result.append({"image_id": img_id.item(), "caption": caption})
            
            for entry in result:
                image_id = entry["image_id"]
                caption = entry["caption"]
                existing_entry = next((item for item in scenario_result if item["image_id"] == image_id), None)
                if existing_entry:
                    existing_entry["caption"].append(caption)
                else:
                    scenario_result.append({"image_id": image_id, "caption": [caption]})
            del model
            torch.cuda.empty_cache()
            print(f"prompt {idj} completed!")
        output_path = model_path.replace("checkpoint_best.pth", "result/adversarial_output.json")
        with open(output_path, 'w') as f:
            json.dump(scenario_result, f, indent=4)
            
        print(f"scenario {idx} completed!")

In [ ]:
pretrained_model = [
        "../output/Caption_flickr_sensitive_dbi/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_edu/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_fi/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_idc/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_mi/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_resume/checkpoint_best.pth"
    ]
get_adversarial_output(pretrained_model, config, filenames, device)

../output/Caption_flickr_sensitive_dbi/checkpoint_best.pth
load checkpoint from ../output/Caption_flickr_sensitive_dbi/checkpoint_best.pth
prompt 0 completed!
load checkpoint from ../output/Caption_flickr_sensitive_dbi/checkpoint_best.pth
prompt 1 completed!
load checkpoint from ../output/Caption_flickr_sensitive_dbi/checkpoint_best.pth
prompt 2 completed!
scenario 0 completed!
../output/Caption_flickr_sensitive_edu/checkpoint_best.pth
load checkpoint from ../output/Caption_flickr_sensitive_edu/checkpoint_best.pth
prompt 0 completed!
load checkpoint from ../output/Caption_flickr_sensitive_edu/checkpoint_best.pth
prompt 1 completed!
load checkpoint from ../output/Caption_flickr_sensitive_edu/checkpoint_best.pth
prompt 2 completed!
scenario 1 completed!
../output/Caption_flickr_sensitive_fi/checkpoint_best.pth
load checkpoint from ../output/Caption_flickr_sensitive_fi/checkpoint_best.pth
prompt 0 completed!
load checkpoint from ../output/Caption_flickr_sensitive_fi/checkpoint_best.pth


In [ ]:
pretrained_model = [
        "../output/Caption_flickr_sensitive_dbi_finetune/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_edu_finetune/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_fi_finetune/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_idc_finetune/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_mi_finetune/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_resume_finetune/checkpoint_best.pth"
    ]
get_adversarial_output(pretrained_model, config, filenames, device)

In [ ]:
pretrained_model = [
        "../output/Caption_flickr_sensitive_dbi_finetune/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_edu_finetune/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_fi_finetune/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_idc_finetune/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_mi_finetune/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_resume_finetune/checkpoint_best.pth"
    ]
get_adversarial_output(pretrained_model, config, filenames, device)